In [1]:
import numpy as np, time, pandas as pd
import faiss, hnswlib
from annoy import AnnoyIndex

K = 10        # jumlah tetangga
NQ = 100      # jumlah query
SEED = 42
METRICS = ['euclidean', 'cosine', 'inner_product']

In [2]:
def normalize(a):
    return a / np.linalg.norm(a, axis=1, keepdims=True)

def ground_truth(X, Q, metric, k=K):
    d = X.shape[1]
    if metric == 'euclidean':
        idx = faiss.IndexFlatL2(d); idx.add(X); return idx.search(Q, k)[1]
    if metric == 'cosine':
        idx = faiss.IndexFlatIP(d); idx.add(normalize(X)); return idx.search(normalize(Q), k)[1]
    idx = faiss.IndexFlatIP(d); idx.add(X); return idx.search(Q, k)[1]

def recall_at_k(pred, gt, k=K):
    return np.mean([len(set(p[:k]) & set(g[:k])) / k for p, g in zip(pred, gt)])

In [3]:
def run_annoy(X, Q, metric, gt):
    f = X.shape[1]
    m = {'euclidean':'euclidean','cosine':'angular','inner_product':'dot'}[metric]
    t0 = time.time()
    idx = AnnoyIndex(f, m)
    for i in range(len(X)): idx.add_item(i, X[i])
    idx.build(10)
    build = time.time() - t0
    t0 = time.time()
    I = [idx.get_nns_by_vector(q, K, search_k=1000) for q in Q]
    qt = (time.time() - t0) * 1000 / len(Q)
    return build, qt, recall_at_k(I, gt)

def run_faiss(X, Q, metric, gt):
    d, n = X.shape[1], len(X)
    Xb, Qb = (normalize(X), normalize(Q)) if metric == 'cosine' else (X, Q)
    mt = faiss.METRIC_L2 if metric == 'euclidean' else faiss.METRIC_INNER_PRODUCT
    nlist = 10 if n <= 1000 else 1000
    t0 = time.time()
    quant = faiss.IndexFlatL2(d) if metric == 'euclidean' else faiss.IndexFlatIP(d)
    idx = faiss.IndexIVFFlat(quant, d, nlist, mt)
    idx.train(Xb); idx.add(Xb)
    build = time.time() - t0
    idx.nprobe = 10
    t0 = time.time()
    I = idx.search(Qb, K)[1]
    qt = (time.time() - t0) * 1000 / len(Q)
    return build, qt, recall_at_k(I, gt)

def run_hnsw(X, Q, metric, gt):
    d, n = X.shape[1], len(X)
    sp = {'euclidean':'l2','cosine':'cosine','inner_product':'ip'}[metric]
    t0 = time.time()
    p = hnswlib.Index(space=sp, dim=d)
    p.init_index(max_elements=n, ef_construction=100, M=16)
    p.add_items(X)
    build = time.time() - t0
    p.set_ef(100)
    t0 = time.time()
    I = p.knn_query(Q, k=K)[0]
    qt = (time.time() - t0) * 1000 / len(Q)
    return build, qt, recall_at_k(I, gt)

In [4]:
configs = [(1000,2),(1000,5),(1_000_000,2),(1_000_000,5)]
rows = []
for n, d in configs:
    rng = np.random.default_rng(SEED)
    X = rng.random((n, d), dtype=np.float32)
    Q = rng.random((NQ, d), dtype=np.float32)
    for metric in METRICS:
        gt = ground_truth(X, Q, metric)
        for name, fn in [('ANNOY', run_annoy), ('FAISS', run_faiss), ('HNSW', run_hnsw)]:
            b, qt, rec = fn(X, Q, metric, gt)
            rows.append(dict(Data=f"{n}/{d}D", Metric=metric, Algoritma=name, Build_s=b, Query_ms=qt, Recall=rec))
            print(f"{n:>8}/{d}D {metric:<14} {name:<6} build={b:8.3f}s query={qt:8.4f}ms recall@{K}={rec:.3f}")
df = pd.DataFrame(rows)

    1000/2D euclidean      ANNOY  build=   0.026s query=  0.2621ms recall@10=1.000
    1000/2D euclidean      FAISS  build=   0.001s query=  0.0043ms recall@10=1.000
    1000/2D euclidean      HNSW   build=   0.040s query=  0.0107ms recall@10=1.000
    1000/2D cosine         ANNOY  build=   0.088s query=  0.1475ms recall@10=1.000
    1000/2D cosine         FAISS  build=   0.001s query=  0.0053ms recall@10=1.000
    1000/2D cosine         HNSW   build=   0.019s query=  0.0123ms recall@10=0.999


    1000/2D inner_product  ANNOY  build=   0.067s query=  0.1925ms recall@10=0.422
    1000/2D inner_product  FAISS  build=   0.001s query=  0.0052ms recall@10=1.000
    1000/2D inner_product  HNSW   build=   0.067s query=  0.0098ms recall@10=0.995
    1000/5D euclidean      ANNOY  build=   0.020s query=  0.1500ms recall@10=1.000
    1000/5D euclidean      FAISS  build=   0.001s query=  0.0042ms recall@10=1.000
    1000/5D euclidean      HNSW   build=   0.078s query=  0.0231ms recall@10=1.000


    1000/5D cosine         ANNOY  build=   0.071s query=  0.1563ms recall@10=1.000
    1000/5D cosine         FAISS  build=   0.001s query=  0.0046ms recall@10=1.000
    1000/5D cosine         HNSW   build=   0.073s query=  0.0246ms recall@10=1.000
    1000/5D inner_product  ANNOY  build=   0.035s query=  0.1417ms recall@10=0.747
    1000/5D inner_product  FAISS  build=   0.046s query=  0.0307ms recall@10=1.000


    1000/5D inner_product  HNSW   build=   0.088s query=  0.0260ms recall@10=0.955


 1000000/2D euclidean      ANNOY  build=  20.929s query=  0.2532ms recall@10=1.000


 1000000/2D euclidean      FAISS  build=   0.801s query=  0.0592ms recall@10=1.000


... (12 outputs left out)

 1000000/5D cosine         HNSW   build=  67.308s query=  0.0326ms recall@10=1.000


 1000000/5D inner_product  ANNOY  build=  29.696s query=  0.1983ms recall@10=0.000


 1000000/5D inner_product  FAISS  build=   9.570s query=  0.1439ms recall@10=0.009


 1000000/5D inner_product  HNSW   build=  59.574s query=  0.0178ms recall@10=0.833


## Tabel hasil


In [5]:
def table(col, fmt):
    for metric in METRICS:
        print(f"\n=== {col} — metric: {metric} ===")
        t = df[df.Metric == metric].pivot(index='Data', columns='Algoritma', values=col)
        t = t.loc[[f"{n}/{d}D" for n, d in configs], ['ANNOY','FAISS','HNSW']]
        t.index.name = 'Jumlah Data/Dimensi'
        display(t.map(fmt.format) if hasattr(t,'map') else t)
table('Recall', '{:.3f}')


=== Recall — metric: euclidean ===


Algoritma,ANNOY,FAISS,HNSW
Jumlah Data/Dimensi,,,
1000/2D,1.000,1.000,1.000
1000/5D,1.000,1.000,1.000
1000000/2D,1.000,1.000,1.000
1000000/5D,1.000,1.000,1.000



=== Recall — metric: cosine ===


Algoritma,ANNOY,FAISS,HNSW
Jumlah Data/Dimensi,,,
1000/2D,1.000,1.000,0.999
1000/5D,1.000,1.000,1.000
1000000/2D,0.063,0.913,0.078
1000000/5D,1.000,1.000,1.000



=== Recall — metric: inner_product ===


Algoritma,ANNOY,FAISS,HNSW
Jumlah Data/Dimensi,,,
1000/2D,0.422,1.000,0.995
1000/5D,0.747,1.000,0.955
1000000/2D,0.000,0.000,0.938
1000000/5D,0.000,0.009,0.833


In [6]:
table('Query_ms', '{:.4f}')


=== Query_ms — metric: euclidean ===


Algoritma,ANNOY,FAISS,HNSW
Jumlah Data/Dimensi,,,
1000/2D,0.2621,0.0043,0.0107
1000/5D,0.1500,0.0042,0.0231
1000000/2D,0.2532,0.0592,0.0407
1000000/5D,0.2820,0.1012,0.0308



=== Query_ms — metric: cosine ===


Algoritma,ANNOY,FAISS,HNSW
Jumlah Data/Dimensi,,,
1000/2D,0.1475,0.0053,0.0123
1000/5D,0.1563,0.0046,0.0246
1000000/2D,0.3445,0.0854,0.0308
1000000/5D,0.2707,0.7183,0.0326



=== Query_ms — metric: inner_product ===


Algoritma,ANNOY,FAISS,HNSW
Jumlah Data/Dimensi,,,
1000/2D,0.1925,0.0052,0.0098
1000/5D,0.1417,0.0307,0.0260
1000000/2D,0.3324,0.0269,0.0119
1000000/5D,0.1983,0.1439,0.0178


In [7]:
table('Build_s', '{:.3f}')


=== Build_s — metric: euclidean ===


Algoritma,ANNOY,FAISS,HNSW
Jumlah Data/Dimensi,,,
1000/2D,0.026,0.001,0.040
1000/5D,0.020,0.001,0.078
1000000/2D,20.929,0.801,47.793
1000000/5D,18.399,1.023,74.497



=== Build_s — metric: cosine ===


Algoritma,ANNOY,FAISS,HNSW
Jumlah Data/Dimensi,,,
1000/2D,0.088,0.001,0.019
1000/5D,0.071,0.001,0.073
1000000/2D,66.063,3.847,77.876
1000000/5D,24.625,10.131,67.308



=== Build_s — metric: inner_product ===


Algoritma,ANNOY,FAISS,HNSW
Jumlah Data/Dimensi,,,
1000/2D,0.067,0.001,0.067
1000/5D,0.035,0.046,0.088
1000000/2D,25.607,5.398,36.323
1000000/5D,29.696,9.570,59.574


## Tabel ringkas gabungan (build s / query ms / recall@10)

In [8]:
df['Hasil'] = df.apply(lambda r: f"{r.Build_s:.2f}s / {r.Query_ms:.3f}ms / {r.Recall:.2f}", axis=1)
for metric in METRICS:
    print(f"\n=== Metric: {metric} ===")
    t = df[df.Metric==metric].pivot(index='Data', columns='Algoritma', values='Hasil')
    t = t.loc[[f"{n}/{d}D" for n,d in configs], ['ANNOY','FAISS','HNSW']]
    t.index.name = 'Jumlah Data/Dimensi'
    display(t)


=== Metric: euclidean ===


Algoritma,ANNOY,FAISS,HNSW
Jumlah Data/Dimensi,,,
1000/2D,0.03s / 0.262ms / 1.00,0.00s / 0.004ms / 1.00,0.04s / 0.011ms / 1.00
1000/5D,0.02s / 0.150ms / 1.00,0.00s / 0.004ms / 1.00,0.08s / 0.023ms / 1.00
1000000/2D,20.93s / 0.253ms / 1.00,0.80s / 0.059ms / 1.00,47.79s / 0.041ms / 1.00
1000000/5D,18.40s / 0.282ms / 1.00,1.02s / 0.101ms / 1.00,74.50s / 0.031ms / 1.00



=== Metric: cosine ===


Algoritma,ANNOY,FAISS,HNSW
Jumlah Data/Dimensi,,,
1000/2D,0.09s / 0.147ms / 1.00,0.00s / 0.005ms / 1.00,0.02s / 0.012ms / 1.00
1000/5D,0.07s / 0.156ms / 1.00,0.00s / 0.005ms / 1.00,0.07s / 0.025ms / 1.00
1000000/2D,66.06s / 0.345ms / 0.06,3.85s / 0.085ms / 0.91,77.88s / 0.031ms / 0.08
1000000/5D,24.62s / 0.271ms / 1.00,10.13s / 0.718ms / 1.00,67.31s / 0.033ms / 1.00



=== Metric: inner_product ===


Algoritma,ANNOY,FAISS,HNSW
Jumlah Data/Dimensi,,,
1000/2D,0.07s / 0.193ms / 0.42,0.00s / 0.005ms / 1.00,0.07s / 0.010ms / 0.99
1000/5D,0.04s / 0.142ms / 0.75,0.05s / 0.031ms / 1.00,0.09s / 0.026ms / 0.96
1000000/2D,25.61s / 0.332ms / 0.00,5.40s / 0.027ms / 0.00,36.32s / 0.012ms / 0.94
1000000/5D,29.70s / 0.198ms / 0.00,9.57s / 0.144ms / 0.01,59.57s / 0.018ms / 0.83


## Tabel Hasil

| Jumlah Data/Dimensi | ANNOY | FAISS | HNSW |
| --- | --- | --- | --- |
| 1000/2D | 0.026s / 0.2621ms / 1.000 | 0.001s / 0.0043ms / 1.000 | 0.040s / 0.0107ms / 1.000 |
| 1000/5D | 0.020s / 0.1500ms / 1.000 | 0.001s / 0.0042ms / 1.000 | 0.078s / 0.0231ms / 1.000 |
| 1000000/2D | 20.929s / 0.2532ms / 1.000 | 0.801s / 0.0592ms / 1.000 | 47.793s / 0.0407ms / 1.000 |
| 1000000/5D | 18.399s / 0.2820ms / 1.000 | 1.023s / 0.1012ms / 1.000 | 74.497s / 0.0308ms / 1.000 |

### Metric: cosine

| Jumlah Data/Dimensi | ANNOY | FAISS | HNSW |
| --- | --- | --- | --- |
| 1000/2D | 0.088s / 0.1475ms / 1.000 | 0.001s / 0.0053ms / 1.000 | 0.019s / 0.0123ms / 0.999 |
| 1000/5D | 0.071s / 0.1563ms / 1.000 | 0.001s / 0.0046ms / 1.000 | 0.073s / 0.0246ms / 1.000 |
| 1000000/2D | 66.063s / 0.3445ms / 0.063 | 3.847s / 0.0854ms / 0.913 | 77.876s / 0.0308ms / 0.078 |
| 1000000/5D | 24.625s / 0.2707ms / 1.000 | 10.131s / 0.7183ms / 1.000 | 67.308s / 0.0326ms / 1.000 |

### Metric: inner_product

| Jumlah Data/Dimensi | ANNOY | FAISS | HNSW |
| --- | --- | --- | --- |
| 1000/2D | 0.067s / 0.1925ms / 0.422 | 0.001s / 0.0052ms / 1.000 | 0.067s / 0.0098ms / 0.995 |
| 1000/5D | 0.035s / 0.1417ms / 0.747 | 0.046s / 0.0307ms / 1.000 | 0.088s / 0.0260ms / 0.955 |
| 1000000/2D | 25.607s / 0.3324ms / 0.000 | 5.398s / 0.0269ms / 0.000 | 36.323s / 0.0119ms / 0.938 |
| 1000000/5D | 29.696s / 0.1983ms / 0.000 | 9.570s / 0.1439ms / 0.009 | 59.574s / 0.0178ms / 0.833 |
